# 🧠 Notebook 2.5 — Estimativa empírica da janela pré-ictal

Parte intermediária do pipeline (entre NB1 e NB2).

**Problema:** rotular o pré-ictal como "10 min antes do onset" é arbitrário e não tem validação.

**Solução:** estimar a duração do pré-ictal **a partir dos dados**, detectando
automaticamente quando a dinâmica do EEG muda do interictal para o pré-ictal,
usando **Change Point Detection (PELT)** + **clustering de validação (K-Means)**.

**Entrada:** arquivos L1 (`data/level1_signals/*.npz`) gerados pelo NB1.

**Saída:** `data/preictal_estimate.json` com a janela estimada (mediana das crises),
que o NB1 e o NB2 vão ler em vez do valor fixo de 10 min.

---
### Método (resumo)
Para cada crise dos pacientes de calibração:
1. Pega o sinal nos **60 min antes do onset ictal** (só usa o onset, que é confiável)
2. Extrai features em janelas de 30 s (região temporal — onde o sinal epiléptico é mais forte)
3. Roda **PELT** sobre a série de features → encontra o changepoint mais próximo do onset
4. A distância (changepoint → onset) = duração estimada do pré-ictal daquela crise
5. **K-Means (K=2)** valida se há 2 regimes distintos (antes/depois da mudança)

Ao final, agrega todas as estimativas: **mediana** (robusta a outliers) vira a janela adotada.


## 1. Imports e configuração

In [ ]:
%pip install -q numpy pandas matplotlib scipy scikit-learn ruptures pywavelets

import os, re, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import ruptures as rpt
import pywt

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Diretórios (mesmos do NB1) ───────────────────────────────────────────────
ROOT_DIR = 'data'
L1_DIR   = os.path.join(ROOT_DIR, 'level1_signals')
OUT_JSON = os.path.join(ROOT_DIR, 'preictal_estimate.json')

# ── Labels (mesmos do NB1) ───────────────────────────────────────────────────
LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)

# ── 5 pacientes de calibração por dataset ────────────────────────────────────
# Use pacientes com gravações COM crise. Estes não devem ser usados no teste final.
CALIB_PATIENTS = {
    'CHBMIT'  : ['chb01', 'chb02', 'chb03', 'chb04', 'chb05'],
    'Siena'   : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06'],
    'Mendeley': ['p10', 'p11', 'p12', 'p13', 'p14'],
    'SeizeIT2': ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005'],
}

# ── Parâmetros do estimador ──────────────────────────────────────────────────
LOOKBACK_SEC   = 60 * 60     # quanto olhar antes do onset (60 min)
FEAT_WIN_SEC   = 30          # tamanho da janela de feature (30 s)
FEAT_STEP_SEC  = 30          # passo entre janelas (sem overlap)
PELT_PENALTY   = 3.0         # penalização do PELT (maior = menos changepoints)
PELT_MIN_SIZE  = 3           # mínimo de janelas entre changepoints (3 = 1.5 min)
MIN_PREICTAL_SEC = 3 * 60    # ignora estimativas < 3 min (provável ruído)
MAX_PREICTAL_SEC = 50 * 60   # trava estimativas absurdas

# Região temporal: canais onde o sinal ictal costuma ser mais evidente
TEMPORAL_HINTS = ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8',
                  'f7','f8']

print('✅ Config OK')
print(f'   Lookback {LOOKBACK_SEC//60}min | janela {FEAT_WIN_SEC}s | PELT pen={PELT_PENALTY}')
for ds, pats in CALIB_PATIENTS.items():
    print(f'   {ds:9}: {pats}')


## 2. Features por janela (compatíveis com o NB2)

In [ ]:
_BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,40)}

def channel_feats(sig, sfreq):
    """
    Mesmas 16 features do NB2 (6 temporais + 6 espectrais + 4 DWT),
    para manter consistência com o restante do pipeline.
    """
    # temporais
    d1 = np.diff(sig); act = float(np.var(sig))
    mob = float(np.sqrt(np.var(d1) / (np.var(sig) + 1e-10)))
    temporal = [float(np.std(sig)), float(np.var(sig)), float(np.sqrt(np.mean(sig**2))),
                float(np.sum(np.abs(d1))), act, mob]
    # espectrais
    nperseg = min(int(sfreq*4), max(int(sfreq), len(sig)//2))
    f, psd = welch(sig, fs=sfreq, nperseg=nperseg)
    bp = []
    for lo,hi in _BANDS.values():
        idx = (f>=lo)&(f<=hi)
        bp.append(float(np.trapz(psd[idx], f[idx])) if idx.sum()>1 else 0.0)
    pn = psd/(psd.sum()+1e-10); pn = pn[pn>0]
    spectral = bp + [float(-np.sum(pn*np.log(pn)))]
    # DWT
    c = pywt.wavedec(sig.astype(np.float64), 'db4', level=4)
    dwt = [float(np.sum(x**2)) for x in c[1:5]]
    return np.array(temporal + spectral + dwt, dtype=np.float32)

def temporal_channel_indices(ch_names):
    """Índices dos canais da região temporal (onde o sinal ictal é mais forte)."""
    idxs = []
    for i, ch in enumerate(ch_names):
        c = str(ch).lower()
        if any(h in c for h in TEMPORAL_HINTS):
            idxs.append(i)
    # fallback: se nenhum temporal, usa todos os canais
    return idxs if idxs else list(range(len(ch_names)))

print('✅ Funções de feature definidas')


## 3. Estimador da janela pré-ictal (PELT + K-Means)

In [ ]:
def extract_preonset_features(data, sfreq, onset_sample, prev_offset_sample=0):
    """
    Extrai a série temporal de features nas janelas ANTES do onset ictal.
    - data: [n_ch, n_samples] (já filtrado pelo NB1)
    - onset_sample: amostra do início da crise
    - prev_offset_sample: fim da crise anterior (não olha antes disso)

    Retorna:
      feats  : (n_janelas, 16)  features médias dos canais temporais
      starts : (n_janelas,)     tempo (s) do início de cada janela, relativo ao onset
                                 (negativo: quanto antes do onset)
    """
    ws   = int(FEAT_WIN_SEC * sfreq)
    step = int(FEAT_STEP_SEC * sfreq)
    look = int(LOOKBACK_SEC * sfreq)

    # início da janela de análise: max(60min antes, fim da crise anterior, 0)
    win_start = max(0, onset_sample - look, prev_offset_sample)
    ch_idx = temporal_channel_indices_cache

    feats, starts = [], []
    for s in range(win_start, onset_sample - ws + 1, step):
        seg = data[ch_idx, s:s+ws]                       # (n_temp_ch, ws)
        # feature média entre os canais temporais
        fvec = np.mean([channel_feats(seg[c], sfreq) for c in range(seg.shape[0])], axis=0)
        feats.append(fvec)
        starts.append((s - onset_sample) / sfreq)        # negativo
    if not feats:
        return np.empty((0,16)), np.empty(0)
    return np.array(feats), np.array(starts)


def estimate_preictal_window(feats, starts):
    """
    Roda PELT na série de features e devolve a duração estimada do pré-ictal.

    Lógica: o changepoint mais próximo do onset marca o início do regime
    "pré-ictal". A distância dele até o onset é a duração estimada.

    Retorna: (dur_sec, cp_time_rel, n_changepoints, silhouette)
             dur_sec = None se não houver pré-ictal detectável confiável
    """
    if len(feats) < 2 * PELT_MIN_SIZE:
        return None, None, 0, None

    # normaliza as features (PELT é sensível a escala)
    Xn = StandardScaler().fit_transform(feats)

    # PELT com modelo RBF (captura mudanças de distribuição multivariada)
    try:
        algo = rpt.Pelt(model='rbf', min_size=PELT_MIN_SIZE).fit(Xn)
        cps  = algo.predict(pen=PELT_PENALTY)   # inclui o índice final (len)
    except Exception:
        return None, None, 0, None

    # remove o índice final (que é sempre len(feats))
    cps = [c for c in cps if c < len(feats)]
    if not cps:
        return None, None, 0, None

    # changepoint mais próximo do onset = início do pré-ictal
    cp_idx   = max(cps)                         # o último (mais perto do fim/onset)
    cp_time  = starts[cp_idx]                   # tempo relativo (negativo)
    dur_sec  = abs(cp_time)                     # duração do pré-ictal

    # validação por clustering: K-Means K=2 (antes vs depois do cp)
    sil = None
    if len(feats) >= 4:
        try:
            km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xn)
            sil = float(silhouette_score(Xn, km.labels_))
        except Exception:
            sil = None

    # filtros de sanidade
    if dur_sec < MIN_PREICTAL_SEC or dur_sec > MAX_PREICTAL_SEC:
        return None, cp_time, len(cps), sil

    return dur_sec, cp_time, len(cps), sil

print('✅ Estimador definido (PELT rbf + K-Means validação)')


## 4. Roda em todos os L1 de calibração

In [ ]:
# Carrega o índice de L1 e filtra pelos pacientes de calibração
l1_files = sorted(glob.glob(os.path.join(L1_DIR, '*_L1.npz')))
print(f'L1 encontrados no disco: {len(l1_files)}')

calib_set = {(ds, p) for ds, pats in CALIB_PATIENTS.items() for p in pats}

results = []   # uma linha por crise
skipped = []

for npz_path in l1_files:
    base = os.path.basename(npz_path)
    # nome: <dataset>__<patient>__<file>_L1.npz
    m = re.match(r'(.+?)__(.+?)__(.+)_L1\.npz', base)
    if not m:
        continue
    ds, pat, fkey = m.group(1), m.group(2), m.group(3)
    if (ds, pat) not in calib_set:
        continue

    npz = np.load(npz_path, allow_pickle=True)
    data   = npz['data']                  # [n_ch, n_samples], já filtrado
    labels = npz['labels']
    sfreq  = float(npz['sfreq'])
    ch     = list(npz['ch_names'])

    # cache dos canais temporais (uma vez por arquivo)
    global temporal_channel_indices_cache
    temporal_channel_indices_cache = temporal_channel_indices(ch)

    # encontra os onsets ictais a partir das labels (transições para ictal)
    is_ictal = (labels == LBL['ictal']).astype(np.int8)
    edges = np.diff(is_ictal)
    onsets  = np.where(edges == 1)[0] + 1
    offsets = np.where(edges == -1)[0] + 1
    if is_ictal[0] == 1:  onsets  = np.r_[0, onsets]
    if is_ictal[-1] == 1: offsets = np.r_[offsets, len(labels)]

    if len(onsets) == 0:
        continue

    for k, onset in enumerate(onsets):
        prev_off = offsets[k-1] if k > 0 else 0
        feats, starts = extract_preonset_features(data, sfreq, onset, prev_off)
        dur, cp_time, n_cp, sil = estimate_preictal_window(feats, starts)

        row = {
            'dataset'        : ds,
            'patient'        : pat,
            'file'           : fkey,
            'seizure_idx'    : k,
            'onset_sec'      : round(onset / sfreq, 1),
            'lookback_avail_min': round((onset - prev_off) / sfreq / 60, 1),
            'n_feat_windows' : len(feats),
            'n_changepoints' : n_cp,
            'silhouette'     : round(sil, 3) if sil is not None else None,
            'preictal_est_min': round(dur / 60, 2) if dur is not None else None,
        }
        if dur is not None:
            results.append(row)
        else:
            skipped.append(row)

df_res  = pd.DataFrame(results)
df_skip = pd.DataFrame(skipped)

print(f'\n✅ Crises com estimativa válida: {len(df_res)}')
print(f'⏭️  Crises descartadas (pré curto/ausente): {len(df_skip)}')
if not df_res.empty:
    import IPython.display as ipd
    ipd.display(df_res)


## 5. Agregação — mediana como janela adotada

In [ ]:
if df_res.empty:
    raise RuntimeError('Nenhuma estimativa válida. Verifique se os L1 de calibração existem.')

est = df_res['preictal_est_min'].values

stats = {
    'n_seizures_used'   : int(len(est)),
    'mean_min'          : round(float(np.mean(est)), 2),
    'median_min'        : round(float(np.median(est)), 2),
    'std_min'           : round(float(np.std(est)), 2),
    'q25_min'           : round(float(np.percentile(est, 25)), 2),
    'q75_min'           : round(float(np.percentile(est, 75)), 2),
    'min_min'           : round(float(np.min(est)), 2),
    'max_min'           : round(float(np.max(est)), 2),
}

# Janela ADOTADA = mediana (robusta a outliers, melhor que média p/ dist. assimétrica)
PRE_SEC_ESTIMATED = int(round(stats['median_min'] * 60))

print('═'*50)
print('  ESTIMATIVA DA JANELA PRÉ-ICTAL')
print('═'*50)
for k, v in stats.items():
    print(f'  {k:18} {v}')
print('─'*50)
print(f'  ➡️  ADOTADO (mediana): {stats["median_min"]} min = {PRE_SEC_ESTIMATED} s')
print('═'*50)

# Por dataset (para comparar)
print('\nPor dataset (mediana):')
for ds, grp in df_res.groupby('dataset'):
    print(f'  {ds:9}: {grp["preictal_est_min"].median():.1f} min  (n={len(grp)} crises)')


In [ ]:
# Salva o resultado para o NB1/NB2 lerem
out = {
    'PRE_SEC_ESTIMATED' : PRE_SEC_ESTIMATED,
    'method'            : 'PELT (rbf) changepoint + KMeans validation',
    'aggregation'       : 'median across calibration seizures',
    'stats'             : stats,
    'calib_patients'    : CALIB_PATIENTS,
    'params'            : {
        'lookback_sec'    : LOOKBACK_SEC,
        'feat_win_sec'    : FEAT_WIN_SEC,
        'pelt_penalty'    : PELT_PENALTY,
        'pelt_min_size'   : PELT_MIN_SIZE,
        'min_preictal_sec': MIN_PREICTAL_SEC,
        'max_preictal_sec': MAX_PREICTAL_SEC,
    },
    'per_seizure'       : df_res.to_dict(orient='records'),
}
with open(OUT_JSON, 'w') as f:
    json.dump(out, f, indent=2)
print(f'💾 Salvo em {OUT_JSON}')
print(f'\n   No NB1 e NB2, substitua:  PRE_SEC = 10*60')
print(f'   por:                      PRE_SEC = json.load(open("{OUT_JSON}"))["PRE_SEC_ESTIMATED"]')


## 6. Visualizações

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1) Histograma das estimativas
ax = axes[0]
ax.hist(est, bins=min(15, len(est)), color='#4C72B0', alpha=0.8, edgecolor='white')
ax.axvline(stats['median_min'], color='#d62728', lw=2, label=f'Mediana {stats["median_min"]}min')
ax.axvline(stats['mean_min'],   color='#ff7f0e', lw=2, ls='--', label=f'Média {stats["mean_min"]}min')
ax.set_title('Distribuição das estimativas', fontweight='bold')
ax.set_xlabel('Janela pré-ictal estimada (min)'); ax.set_ylabel('Nº de crises')
ax.legend(fontsize=8); ax.spines[['top','right']].set_visible(False)

# 2) Boxplot por dataset
ax = axes[1]
ds_order = [d for d in ['CHBMIT','Siena','Mendeley','SeizeIT2'] if d in df_res['dataset'].unique()]
data_by_ds = [df_res[df_res['dataset']==d]['preictal_est_min'].values for d in ds_order]
bp = ax.boxplot(data_by_ds, labels=ds_order, patch_artist=True,
                medianprops=dict(color='black', lw=2))
cmap = {'CHBMIT':'#4C72B0','Siena':'#DD8452','Mendeley':'#55A868','SeizeIT2':'#C44E52'}
for patch, d in zip(bp['boxes'], ds_order):
    patch.set_facecolor(cmap[d]); patch.set_alpha(0.7)
ax.set_title('Estimativa por dataset', fontweight='bold')
ax.set_ylabel('Janela pré-ictal (min)'); ax.tick_params(axis='x', rotation=20)
ax.spines[['top','right']].set_visible(False)

# 3) Silhouette vs estimativa (qualidade da separação)
ax = axes[2]
valid = df_res.dropna(subset=['silhouette'])
if not valid.empty:
    sc = ax.scatter(valid['preictal_est_min'], valid['silhouette'],
                    c=[cmap[d] for d in valid['dataset']], s=60, alpha=0.8, edgecolors='white')
    ax.set_title('Qualidade da separação (K-Means)', fontweight='bold')
    ax.set_xlabel('Janela estimada (min)'); ax.set_ylabel('Silhouette (↑ melhor)')
    ax.axhline(0.5, color='gray', ls=':', label='silhouette 0.5')
    ax.legend(fontsize=8); ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(ROOT_DIR, 'preictal_estimate.png'), dpi=140, bbox_inches='tight')
plt.show()
print('💾 Gráfico salvo em', os.path.join(ROOT_DIR, 'preictal_estimate.png'))


## 7. Exemplo: changepoint em uma crise individual

In [ ]:
# Mostra, para a primeira crise válida, a série de features e o changepoint detectado
if not df_res.empty:
    ex = df_res.iloc[0]
    npz_path = glob.glob(os.path.join(L1_DIR, f'{ex.dataset}__{ex.patient}__*_L1.npz'))
    npz_path = [p for p in npz_path if ex.file in p]
    if npz_path:
        npz = np.load(npz_path[0], allow_pickle=True)
        data = npz['data']; labels = npz['labels']; sfreq = float(npz['sfreq']); ch = list(npz['ch_names'])
        temporal_channel_indices_cache = temporal_channel_indices(ch)

        is_ictal = (labels == LBL['ictal']).astype(np.int8)
        onsets = np.where(np.diff(is_ictal) == 1)[0] + 1
        if is_ictal[0]==1: onsets = np.r_[0, onsets]
        onset = onsets[int(ex.seizure_idx)]

        feats, starts = extract_preonset_features(data, sfreq, onset, 0)
        Xn = StandardScaler().fit_transform(feats)
        algo = rpt.Pelt(model='rbf', min_size=PELT_MIN_SIZE).fit(Xn)
        cps = [c for c in algo.predict(pen=PELT_PENALTY) if c < len(feats)]

        fig, ax = plt.subplots(figsize=(12, 4))
        # plota algumas features normalizadas
        starts_min = starts / 60
        for j, name in zip([0, 6, 11], ['energia (std)', 'banda delta', 'entropia espectral']):
            ax.plot(starts_min, Xn[:, j], lw=1.2, alpha=0.8, label=name)
        for c in cps:
            ax.axvline(starts_min[c], color='#d62728', ls='--', alpha=0.7)
        if cps:
            cp_best = max(cps)
            ax.axvline(starts_min[cp_best], color='#d62728', lw=2.5,
                       label=f'CP adotado (-{abs(starts_min[cp_best]):.1f}min)')
        ax.axvline(0, color='black', lw=2, label='onset ictal')
        ax.set_title(f'{ex.dataset} / {ex.patient} / crise {int(ex.seizure_idx)} — '
                     f'pré-ictal estimado: {ex.preictal_est_min} min',
                     fontweight='bold')
        ax.set_xlabel('Tempo relativo ao onset (min)'); ax.set_ylabel('Feature (z-score)')
        ax.legend(fontsize=8, loc='upper left'); ax.spines[['top','right']].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(ROOT_DIR, 'preictal_example.png'), dpi=140, bbox_inches='tight')
        plt.show()
        print('💾 Exemplo salvo em', os.path.join(ROOT_DIR, 'preictal_example.png'))


---
✅ **Fim do Notebook 2.5.**

A janela pré-ictal estimada está em `data/preictal_estimate.json` (campo `PRE_SEC_ESTIMATED`).

**Próximo passo:** ajuste o NB1 (célula da função `build_label_array`) para ler esse valor
em vez do `PRE_SEC = 10*60` fixo, e re-execute o NB1 → NB2 → NB3.
